In [1]:
#| default_exp models.auto_arima

In [1]:
#| export
from __future__ import annotations
from typing import List, Dict, Optional, Callable, Tuple, Any, Union
import time
import copy
import itertools
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import kpss, adfuller

from peshbeen.transformations import box_cox_transform
from peshbeen.model_selection import SplitTimeSeries
from peshbeen.models.arima import arima


#| export
def check_stationarity_d(
    y: np.ndarray,
    max_d: int = 2,
    alpha: float = 0.05,
    test: str = "kpss",
) -> Tuple[int, List[Dict[str, Any]]]:
    """
    Determine the integration order d using repeated stationarity tests (Hyndman-Khandakar Step 1).

    Parameters
    ----------
    y : np.ndarray
        1D target time series array.
    max_d : int, default 2
        Maximum differencing order to consider.
    alpha : float, default 0.05
        Significance level for the stationarity hypothesis test.
    test : str, default 'kpss'
        Stationarity test to apply: 'kpss' (Kwiatkowski-Phillips-Schmidt-Shin) or 'adf' (Augmented Dickey-Fuller).

    Returns
    -------
    d : int
        The determined differencing order (0 <= d <= max_d).
    logs : list of dict
        Diagnostics and test statistics for each differencing iteration.
    """
    y_curr = np.asarray(y, dtype=np.float64)
    y_curr = y_curr[~np.isnan(y_curr)]
    if len(y_curr) < 4:
        return 0, [{"d": 0, "stat": np.nan, "pval": 1.0, "action": "too_short"}]

    d = 0
    logs = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        while d < max_d:
            # Check constant series
            if np.all(y_curr == y_curr[0]):
                logs.append({"d": d, "stat": 0.0, "pval": 1.0, "action": "constant_series"})
                break

            if test == "kpss":
                # Null hypothesis H0: Series is level stationary
                try:
                    stat, pval, _, _ = kpss(y_curr, regression="c", nlags="auto")
                    is_stationary = (pval >= alpha)
                except Exception as ex:
                    logs.append({"d": d, "stat": np.nan, "pval": np.nan, "action": f"kpss_error_{ex}"})
                    break
            elif test == "adf":
                # Null hypothesis H0: Series has unit root (non-stationary)
                try:
                    res = adfuller(y_curr)
                    stat, pval = res[0], res[1]
                    is_stationary = (pval < alpha)
                except Exception as ex:
                    logs.append({"d": d, "stat": np.nan, "pval": np.nan, "action": f"adf_error_{ex}"})
                    break
            else:
                raise ValueError(f"Unsupported stationarity test '{test}'. Use 'kpss' or 'adf'.")

            if is_stationary:
                logs.append({
                    "d": d,
                    "stat": float(stat),
                    "pval": float(pval),
                    "action": "stationary_stop"
                })
                break
            else:
                logs.append({
                    "d": d,
                    "stat": float(stat),
                    "pval": float(pval),
                    "action": "difference"
                })
                y_curr = np.diff(y_curr)
                d += 1

        if d == max_d and (len(logs) == 0 or logs[-1]["action"] == "difference"):
            # Max d reached
            try:
                if test == "kpss":
                    stat, pval, _, _ = kpss(y_curr, regression="c", nlags="auto")
                else:
                    res = adfuller(y_curr)
                    stat, pval = res[0], res[1]
            except Exception:
                stat, pval = np.nan, np.nan
            logs.append({
                "d": d,
                "stat": float(stat) if not np.isnan(stat) else np.nan,
                "pval": float(pval) if not np.isnan(pval) else np.nan,
                "action": "max_d_reached"
            })

    return d, logs


#| export
class auto_arima:
    """
    Automatic ARIMA / SARIMA model selection implementing the Hyndman-Khandakar (2008)
    stepwise search algorithm and exhaustive grid search under the unified peshbeen pipeline.

    The algorithm determines integration order d via repeated KPSS stationarity tests,
    evaluates initial candidate models, and performs a stepwise traversal of neighboring
    (p, q, P, Q, D) variations to minimize the sample-size corrected AICc (or AIC, BIC).
    Exogenous regressors, Box-Cox transformations, lag transforms, and trend handling
    are fully supported and evaluated within the native arima engine.
    """

    def __init__(
        self,
        target_col: str,
        seasonal_length: Optional[int] = None,
        d: Optional[int] = None,
        D: Optional[int] = None,
        max_p: int = 5,
        max_q: int = 5,
        max_P: int = 2,
        max_Q: int = 2,
        max_d: int = 2,
        max_D: int = 1,
        start_p: int = 2,
        start_q: int = 2,
        start_P: int = 1,
        start_Q: int = 1,
        max_order: Optional[int] = None,
        stepwise: bool = True,
        criterion: str = "aicc",
        stationarity_test: str = "kpss",
        alpha: float = 0.05,
        with_intercept: Optional[bool] = None,
        lag_transform: Optional[list] = None,
        trend: Optional[str] = None,
        pol_degree: int = 1,
        ets_params: Optional[Dict[str, Any]] = None,
        change_points: Optional[List[int]] = None,
        box_cox: Union[bool, float, int] = False,
        box_cox_biasadj: bool = False,
        cat_variables: Optional[List[str]] = None,
        categorical_encoder: Optional[Any] = None,
        target_encode: bool = False,
        maxiter: int = 200,
        tol: float = 1e-7,
        trace: bool = False,
        **kwargs
    ) -> None:
        """
        Initialize auto_arima model selection.

        Parameters
        ----------
        target_col : str
            Name of the target column in the input DataFrame.
        seasonal_length : int, optional
            Seasonal period length (e.g. 7 for daily with weekly pattern, 12 for monthly with annual pattern).
            Default is None (non-seasonal).
        d : int, optional
            Integration/differencing order. If None (default), determined automatically via repeated
            stationarity testing (KPSS).
        D : int, optional
            Seasonal differencing order. If None (default), considered in the model search space (0 to max_D).
        max_p : int, default 5
            Maximum non-seasonal AR order.
        max_q : int, default 5
            Maximum non-seasonal MA order.
        max_P : int, default 2
            Maximum seasonal AR order.
        max_Q : int, default 2
            Maximum seasonal MA order.
        max_d : int, default 2
            Maximum non-seasonal differencing order tested during stationarity check.
        max_D : int, default 1
            Maximum seasonal differencing order explored in search space when D is None.
        start_p : int, default 2
            Initial p order for Hyndman-Khandakar stepwise search.
        start_q : int, default 2
            Initial q order for Hyndman-Khandakar stepwise search.
        start_P : int, default 1
            Initial P order for seasonal Hyndman-Khandakar stepwise search.
        start_Q : int, default 1
            Initial Q order for seasonal Hyndman-Khandakar stepwise search.
        max_order : int, optional
            Maximum sum of orders (p + q + P + Q). If None, no cumulative constraint is applied.
        stepwise : bool, default True
            Whether to use the Hyndman-Khandakar stepwise search algorithm. If False, an exhaustive
            grid search over all combinations within max orders is performed.
        criterion : str, default 'aicc'
            Information criterion to minimize ('aicc', 'aic', or 'bic').
        stationarity_test : str, default 'kpss'
            Statistical test used to determine d ('kpss' or 'adf').
        alpha : float, default 0.05
            Significance level for the stationarity test.
        with_intercept : bool, optional
            Whether to include a constant / drift term. If None (default), constant is tested according
            to Hyndman-Khandakar rules (included unless d + D >= 2, and evaluated in variations).
        lag_transform : list of callable, optional
            List of lag transform functions for exogenous feature generation.
        trend : str, optional
            Deterministic trend removal strategy ('linear' or 'ets').
        pol_degree : int, default 1
            Degree of polynomial trend when trend='linear'.
        ets_params : dict, optional
            Parameters for ETS trend model when trend='ets'.
        change_points : list of int, optional
            Change point indices for piecewise linear trend.
        box_cox : bool or float or int, default False
            Whether to apply Box-Cox transformation (or fixed lambda value).
        box_cox_biasadj : bool, default False
            Whether to apply bias adjustment when back-transforming Box-Cox forecasts.
        cat_variables : list of str, optional
            List of categorical covariate column names.
        categorical_encoder : object, optional
            Categorical encoder instance (e.g. OneHotEncoder).
        target_encode : bool, default False
            Whether target encoding is applied.
        maxiter : int, default 200
            Maximum L-BFGS-B iterations per candidate ARIMA model.
        tol : float, default 1e-7
            Convergence tolerance for optimizer.
        trace : bool, default False
            Whether to print candidate models and their criterion scores during search.
        **kwargs :
            Additional parameter aliases (e.g. maxp, maxq, maxP, maxQ, s, include_constant).
        """
        # Handle aliases
        if "maxp" in kwargs:
            max_p = kwargs.pop("maxp")
        if "maxq" in kwargs:
            max_q = kwargs.pop("maxq")
        if "maxP" in kwargs:
            max_P = kwargs.pop("maxP")
        if "maxQ" in kwargs:
            max_Q = kwargs.pop("maxQ")
        if "maxd" in kwargs:
            max_d = kwargs.pop("maxd")
        if "maxD" in kwargs:
            max_D = kwargs.pop("maxD")
        if "s" in kwargs:
            seasonal_length = kwargs.pop("s")
        if "include_constant" in kwargs:
            with_intercept = kwargs.pop("include_constant")

        self.target_col = target_col
        self.seasonal_length = seasonal_length
        self.s = seasonal_length if (seasonal_length is not None and seasonal_length > 1) else 0

        self.d = d
        self.D = D
        self.max_p = int(max_p)
        self.max_q = int(max_q)
        self.max_P = int(max_P) if self.s > 1 else 0
        self.max_Q = int(max_Q) if self.s > 1 else 0
        self.max_d = int(max_d)
        self.max_D = int(max_D) if self.s > 1 else 0

        self.start_p = min(int(start_p), self.max_p)
        self.start_q = min(int(start_q), self.max_q)
        self.start_P = min(int(start_P), self.max_P) if self.s > 1 else 0
        self.start_Q = min(int(start_Q), self.max_Q) if self.s > 1 else 0

        self.max_order = max_order
        self.stepwise = stepwise
        self.criterion = criterion.lower()
        if self.criterion not in ["aicc", "aic", "bic"]:
            raise ValueError(f"criterion must be 'aicc', 'aic', or 'bic', got '{criterion}'")
        self.stationarity_test = stationarity_test.lower()
        self.alpha = float(alpha)
        self.with_intercept = with_intercept

        self.lag_transform = lag_transform
        self.trend = trend
        self.pol_degree = pol_degree
        self.ets_params = ets_params
        self.change_points = change_points
        self.box_cox = box_cox
        self.box_cox_biasadj = box_cox_biasadj
        self.cat_variables = cat_variables
        self.categorical_encoder = categorical_encoder
        self.target_encode = target_encode
        self.maxiter = maxiter
        self.tol = tol
        self.trace = trace
        self.extra_kwargs = kwargs

        # Fit placeholders
        self.best_model_: Optional[arima] = None
        self.order_: Optional[Tuple[int, int, int]] = None
        self.seasonal_order_: Optional[Tuple[int, int, int]] = None
        self.include_constant_: bool = False
        self.criterion_value_: float = np.inf
        self.search_results_: Optional[pd.DataFrame] = None
        self.stationarity_logs_: List[Dict[str, Any]] = []
        self.total_search_time_: float = 0.0
        self.n_models_evaluated_: int = 0
        self.d_: Optional[int] = None
        self.p_: Optional[int] = None
        self.q_: Optional[int] = None
        self.P_: Optional[int] = None
        self.D_: Optional[int] = None
        self.Q_: Optional[int] = None

        # Delegated placeholders
        self.fitted_values: Optional[np.ndarray] = None
        self.residuals_: Optional[np.ndarray] = None
        self.actuals: Optional[np.ndarray] = None
        self.in_samp_resids: Optional[np.ndarray] = None
        self.params_: Optional[np.ndarray] = None
        self.param_names_: List[str] = []
        self.sigma2_: Optional[float] = None
        self.intercept_: float = 0.0
        self.beta_: Optional[np.ndarray] = None
        self.cv_summary: Optional[pd.DataFrame] = None

    def _eval_model(
        self,
        df: pd.DataFrame,
        p: int,
        d: int,
        q: int,
        P: int,
        D: int,
        Q: int,
        inc_c: bool,
        evaluated: Dict[Tuple[int, int, int, int, int, int, bool], Tuple[float, Optional[arima]]],
        records: List[Dict[str, Any]],
    ) -> Tuple[float, Optional[arima]]:
        """Fit a single ARIMA candidate, record performance, and update model cache."""
        key = (p, d, q, P, D, Q, inc_c)
        if key in evaluated:
            return evaluated[key]

        # Enforce bounds
        if not (0 <= p <= self.max_p and 0 <= q <= self.max_q):
            evaluated[key] = (np.inf, None)
            return np.inf, None
        if self.s > 1:
            if not (0 <= P <= self.max_P and 0 <= Q <= self.max_Q and 0 <= D <= self.max_D):
                evaluated[key] = (np.inf, None)
                return np.inf, None
        else:
            if P != 0 or Q != 0 or D != 0:
                evaluated[key] = (np.inf, None)
                return np.inf, None

        # Cumulative max order constraint
        if self.max_order is not None and (p + q + P + Q > self.max_order):
            evaluated[key] = (np.inf, None)
            return np.inf, None

        score = np.inf
        fitted_mod = None
        fit_time = 0.0

        try:
            cand = arima(
                target_col=self.target_col,
                order=(p, d, q),
                seasonal_order=(P, D, Q) if self.s > 1 else (0, 0, 0),
                seasonal_length=self.s if self.s > 1 else None,
                lag_transform=self.lag_transform,
                trend=self.trend,
                pol_degree=self.pol_degree,
                ets_params=self.ets_params,
                change_points=self.change_points,
                box_cox=self.box_cox,
                box_cox_biasadj=self.box_cox_biasadj,
                cat_variables=self.cat_variables,
                categorical_encoder=self.categorical_encoder,
                target_encode=self.target_encode,
                include_constant=inc_c,
                maxiter=self.maxiter,
                tol=self.tol
            )
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                cand.fit(df)

            fit_time = cand.fit_time_
            if self.criterion == "aicc":
                val = cand.aicc
            elif self.criterion == "aic":
                val = cand.aic
            elif self.criterion == "bic":
                val = cand.bic
            else:
                val = cand.aicc

            if val is not None and not np.isnan(val) and not np.isinf(val) and val > -1e15:
                score = float(val)
                fitted_mod = cand
        except Exception:
            score = np.inf
            fitted_mod = None

        evaluated[key] = (score, fitted_mod)
        records.append({
            "p": p,
            "d": d,
            "q": q,
            "P": P,
            "D": D,
            "Q": Q,
            "s": self.s,
            "constant": inc_c,
            "criterion": self.criterion,
            "score": score,
            "aic": getattr(fitted_mod, "aic", np.nan) if fitted_mod else np.nan,
            "aicc": getattr(fitted_mod, "aicc", np.nan) if fitted_mod else np.nan,
            "bic": getattr(fitted_mod, "bic", np.nan) if fitted_mod else np.nan,
            "fit_time": fit_time,
        })

        if self.trace:
            seas_repr = f"({P},{D},{Q})[{self.s}]" if self.s > 1 else ""
            c_str = f"constant={inc_c}"
            score_str = f"{score:.3f}" if score < np.inf else "inf/failed"
            print(f"Model ARIMA({p},{d},{q}){seas_repr} {c_str:<16} : {self.criterion.upper()} = {score_str}")

        return score, fitted_mod

    def fit(self, df: pd.DataFrame) -> "auto_arima":
        """
        Fit auto_arima on the input time series DataFrame.

        Conducts repeated stationarity tests to determine d, initializes the candidate models,
        and executes stepwise or grid search to select the model minimizing the criterion.

        Parameters
        ----------
        df : pd.DataFrame
            Input DataFrame containing target_col and any exogenous regressors.

        Returns
        -------
        self : auto_arima
            Fitted auto_arima model instance.
        """
        if self.target_col not in df.columns:
            raise ValueError(f"target_col '{self.target_col}' not found in df. Available: {list(df.columns)}")

        start_time = time.time()
        y_raw = df[self.target_col].values.astype(np.float64)

        # ── Step 1: Stationarity Check for d ──────────────────────────────────
        if self.d is None:
            # Transform if Box-Cox is requested for stationarity test
            y_stat = y_raw
            if self.box_cox is not False and np.all(y_raw > 0):
                try:
                    fixed_lam = self.box_cox if isinstance(self.box_cox, (float, int)) and not isinstance(self.box_cox, bool) else None
                    y_stat, _ = box_cox_transform(y_raw, lamda=fixed_lam)
                except Exception:
                    y_stat = y_raw
            self.d_, self.stationarity_logs_ = check_stationarity_d(
                y_stat, max_d=self.max_d, alpha=self.alpha, test=self.stationarity_test
            )
        else:
            self.d_ = int(self.d)
            self.stationarity_logs_ = [{"d": self.d_, "action": "user_specified_d"}]

        d = self.d_
        s = self.s

        # ── Determine D Search Space ──────────────────────────────────────────
        if s > 1:
            if self.D is None:
                D_space = list(range(0, self.max_D + 1))
                start_D = 1 if self.max_D >= 1 else 0
                search_D = True
            else:
                D_space = [int(self.D)]
                start_D = int(self.D)
                search_D = False
        else:
            D_space = [0]
            start_D = 0
            search_D = False

        evaluated: Dict[Tuple[int, int, int, int, int, int, bool], Tuple[float, Optional[arima]]] = {}
        records: List[Dict[str, Any]] = []

        # ── Step 2: Stepwise Search (Hyndman-Khandakar) ───────────────────────
        if self.stepwise:
            # Stage 2(a): Initial models
            # 1. ARIMA(0, d, 0)
            # 2. ARIMA(2, d, 2)
            # 3. ARIMA(1, d, 0)
            # 4. ARIMA(0, d, 1)
            # Constant included unless d + start_D >= 2.
            # If d + start_D <= 1, additional ARIMA(0, d, 0) without constant.

            init_candidates: List[Tuple[int, int, int, int, int, int, bool]] = []
            cand_p2 = min(self.start_p, self.max_p)
            cand_q2 = min(self.start_q, self.max_q)
            cand_P = min(self.start_P, self.max_P) if s > 1 else 0
            cand_Q = min(self.start_Q, self.max_Q) if s > 1 else 0

            # Default constant flag
            if self.with_intercept is not None:
                default_c = bool(self.with_intercept)
            else:
                default_c = bool((d + start_D) < 2)

            # Core 4 models
            init_candidates.append((0, d, 0, 0, start_D, 0, default_c))
            init_candidates.append((cand_p2, d, cand_q2, cand_P, start_D, cand_Q, default_c))
            init_candidates.append((min(1, self.max_p), d, 0, cand_P, start_D, 0, default_c))
            init_candidates.append((0, d, min(1, self.max_q), 0, start_D, cand_Q, default_c))

            # Additional model without constant if d + start_D <= 1 and with_intercept is None
            if self.with_intercept is None and (d + start_D <= 1) and default_c:
                init_candidates.append((0, d, 0, 0, start_D, 0, False))

            # If D is in search space (D was None), evaluate initial models for both D=0 and D=1
            if search_D and len(D_space) > 1:
                alt_D = 0 if start_D == 1 else 1
                alt_c = bool((d + alt_D) < 2) if self.with_intercept is None else bool(self.with_intercept)
                init_candidates.append((0, d, 0, 0, alt_D, 0, alt_c))
                init_candidates.append((cand_p2, d, cand_q2, cand_P, alt_D, cand_Q, alt_c))

            # Fit all initial models
            best_score = np.inf
            best_cand: Optional[Tuple[int, int, int, int, int, int, bool]] = None
            best_model: Optional[arima] = None

            for p_c, d_c, q_c, P_c, D_c, Q_c, c_c in init_candidates:
                sc, mod = self._eval_model(df, p_c, d_c, q_c, P_c, D_c, Q_c, c_c, evaluated, records)
                if sc < best_score:
                    best_score = sc
                    best_cand = (p_c, d_c, q_c, P_c, D_c, Q_c, c_c)
                    best_model = mod

            # If none of initial models succeeded, fallback to (0, d, 0, 0, 0, 0, False)
            if best_cand is None:
                best_cand = (0, d, 0, 0, start_D, 0, False)
                best_score, best_model = self._eval_model(df, 0, d, 0, 0, start_D, 0, False, evaluated, records)

            # Stage 2(c): Stepwise neighborhood search
            current_cand = best_cand
            while True:
                curr_p, curr_d, curr_q, curr_P, curr_D, curr_Q, curr_c = current_cand

                # Generate neighborhood variations
                variations = set()

                # p and q variations (+-1)
                for dp, dq in [
                    (1, 0), (-1, 0), (0, 1), (0, -1),
                    (1, 1), (1, -1), (-1, 1), (-1, -1)
                ]:
                    np_p = curr_p + dp
                    nq_q = curr_q + dq
                    if 0 <= np_p <= self.max_p and 0 <= nq_q <= self.max_q:
                        variations.add((np_p, curr_d, nq_q, curr_P, curr_D, curr_Q, curr_c))

                # Seasonal P and Q variations (+-1) if s > 1
                if s > 1:
                    for dP, dQ in [
                        (1, 0), (-1, 0), (0, 1), (0, -1),
                        (1, 1), (1, -1), (-1, 1), (-1, -1)
                    ]:
                        nP_P = curr_P + dP
                        nQ_Q = curr_Q + dQ
                        if 0 <= nP_P <= self.max_P and 0 <= nQ_Q <= self.max_Q:
                            variations.add((curr_p, curr_d, curr_q, nP_P, curr_D, nQ_Q, curr_c))

                    # If D was None, explore D +- 1
                    if search_D:
                        for dD in [1, -1]:
                            nD_D = curr_D + dD
                            if 0 <= nD_D <= self.max_D:
                                n_c = curr_c if (curr_d + nD_D < 2 or not curr_c) else False
                                variations.add((curr_p, curr_d, curr_q, curr_P, nD_D, curr_Q, n_c))

                # Include / exclude constant variation
                if self.with_intercept is None:
                    toggle_c = not curr_c
                    # Exclude constant if d + D >= 2 unless user explicitly allows
                    if not (toggle_c and (curr_d + curr_D >= 2)):
                        variations.add((curr_p, curr_d, curr_q, curr_P, curr_D, curr_Q, toggle_c))

                # Remove already evaluated
                unevaluated = [v for v in variations if v not in evaluated]
                if not unevaluated:
                    break

                # Evaluate all neighbors
                improved = False
                for cand_v in unevaluated:
                    sc, mod = self._eval_model(df, *cand_v, evaluated, records)
                    if sc < best_score:
                        best_score = sc
                        best_cand = cand_v
                        best_model = mod
                        improved = True

                if improved:
                    current_cand = best_cand
                else:
                    # No neighbor improved the score: Stepwise traversal converged
                    break

        # ── Step 3: Exhaustive Grid Search (stepwise=False) ───────────────────
        else:
            p_range = list(range(0, self.max_p + 1))
            q_range = list(range(0, self.max_q + 1))
            P_range = list(range(0, self.max_P + 1)) if s > 1 else [0]
            Q_range = list(range(0, self.max_Q + 1)) if s > 1 else [0]

            best_score = np.inf
            best_cand = None
            best_model = None

            for p_val in p_range:
                for q_val in q_range:
                    for D_val in D_space:
                        for P_val in P_range:
                            for Q_val in Q_range:
                                # Determine constant candidates
                                if self.with_intercept is not None:
                                    c_candidates = [bool(self.with_intercept)]
                                else:
                                    if d + D_val < 2:
                                        c_candidates = [True, False]
                                    else:
                                        c_candidates = [False]

                                for c_val in c_candidates:
                                    sc, mod = self._eval_model(
                                        df, p_val, d, q_val, P_val, D_val, Q_val, c_val, evaluated, records
                                    )
                                    if sc < best_score:
                                        best_score = sc
                                        best_cand = (p_val, d, q_val, P_val, D_val, Q_val, c_val)
                                        best_model = mod

        # Fallback if every model failed
        if best_model is None:
            # Fit minimal baseline (0, d, 0)
            best_cand = (0, d, 0, 0, start_D, 0, False)
            best_score, best_model = self._eval_model(df, 0, d, 0, 0, start_D, 0, False, evaluated, records)

        self.total_search_time_ = time.time() - start_time
        self.n_models_evaluated_ = len(evaluated)

        # Store chosen parameters
        bp, bd, bq, bP, bD, bQ, bc = best_cand
        self.best_model_ = best_model
        self.order_ = (bp, bd, bq)
        self.seasonal_order_ = (bP, bD, bQ)
        self.include_constant_ = bc
        self.criterion_value_ = best_score
        self.p_, self.d_, self.q_ = self.order_
        self.P_, self.D_, self.Q_ = self.seasonal_order_

        # Create search results DataFrame sorted by score
        self.search_results_ = pd.DataFrame(records).sort_values("score", ascending=True).reset_index(drop=True)

        # Delegate fitted attributes from best_model_
        if self.best_model_ is not None:
            self.fitted_values = self.best_model_.fitted_values
            self.residuals_ = self.best_model_.residuals_
            self.actuals = self.best_model_.actuals
            self.in_samp_resids = self.best_model_.in_samp_resids
            self.params_ = self.best_model_.params_
            self.param_names_ = self.best_model_.param_names_
            self.sigma2_ = self.best_model_.sigma2_
            self.intercept_ = self.best_model_.intercept_
            self.beta_ = self.best_model_.beta_

        return self

    def forecast(
        self,
        H: int,
        exog: Optional[pd.DataFrame] = None
    ) -> np.ndarray:
        """
        Generate recursive multi-step out-of-sample forecasts using the best selected model.

        Parameters
        ----------
        H : int
            Forecast horizon length.
        exog : pd.DataFrame, optional
            Future exogenous features of length H. Required if exogenous regressors were present during training.

        Returns
        -------
        np.ndarray
            1D array of length H containing point forecasts.
        """
        if self.best_model_ is None:
            raise ValueError("auto_arima has not been fitted yet. Call .fit(df) before .forecast(H).")
        return self.best_model_.forecast(H=H, exog=exog)

    def predict_in_sample(self) -> np.ndarray:
        """
        Return in-sample fitted values on the training data.

        Returns
        -------
        np.ndarray
            Fitted in-sample values.
        """
        if self.best_model_ is None:
            raise ValueError("auto_arima has not been fitted yet. Call .fit(df) before .predict_in_sample().")
        return self.best_model_.predict_in_sample()

    @property
    def aic(self) -> float:
        """Akaike Information Criterion of best selected model."""
        return self.best_model_.aic if self.best_model_ is not None else np.nan

    @property
    def aicc(self) -> float:
        """Corrected Akaike Information Criterion of best selected model."""
        return self.best_model_.aicc if self.best_model_ is not None else np.nan

    @property
    def bic(self) -> float:
        """Bayesian Information Criterion of best selected model."""
        return self.best_model_.bic if self.best_model_ is not None else np.nan

    @property
    def hqc(self) -> float:
        """Hannan-Quinn Information Criterion of best selected model."""
        return self.best_model_.hqc if self.best_model_ is not None else np.nan

    def summary(self) -> str:
        """
        Return a comprehensive summary of the model selection process and statistical estimation results.
        """
        if self.best_model_ is None:
            return "auto_arima: Model not fitted yet. Call .fit(df)."

        lines = []
        lines.append("=" * 86)
        lines.append("                  Auto ARIMA Model Selection Summary")
        lines.append("=" * 86)
        method_str = "Hyndman-Khandakar Stepwise Search" if self.stepwise else "Exhaustive Grid Search"
        lines.append(f"Selection Algorithm:    {method_str}")
        lines.append(f"Stationarity Test:      {self.stationarity_test.upper()} (alpha = {self.alpha}) -> d = {self.d_}")
        seas_str = f"({self.P_},{self.D_},{self.Q_})[{self.s}]" if self.s > 1 else "Non-seasonal"
        lines.append(f"Selected Best Model:    ARIMA({self.p_},{self.d_},{self.q_}) {seas_str}")
        lines.append(f"Include Constant:       {self.include_constant_}")
        lines.append(f"Optimized Criterion:    {self.criterion.upper()} = {self.criterion_value_:.4f}")
        lines.append(f"Information Criteria:   AIC = {self.aic:.4f} | AICc = {self.aicc:.4f} | BIC = {self.bic:.4f}")
        lines.append(f"Models Evaluated:       {self.n_models_evaluated_} models")
        lines.append("-" * 86)
        lines.append("Best Model Parameter Estimates:")
        lines.append(self.best_model_.summary())
        return "\n".join(lines)

    def top_models(self, n: int = 5) -> pd.DataFrame:
        """
        Return the top n candidate models evaluated during the search ordered by the optimization criterion.

        Parameters
        ----------
        n : int, default 5
            Number of top models to report.

        Returns
        -------
        pd.DataFrame
            DataFrame containing model specifications, constant indicator, and criterion scores.
        """
        if self.search_results_ is None or len(self.search_results_) == 0:
            raise ValueError("Model has not been fitted yet. Call .fit(df) before calling top_models().")

        top_df = self.search_results_.head(n).copy()
        model_names = []
        for _, r in top_df.iterrows():
            if r["s"] > 1:
                name = f"ARIMA({int(r['p'])},{int(r['d'])},{int(r['q'])})({int(r['P'])},{int(r['D'])},{int(r['Q'])})[{int(r['s'])}]"
            else:
                name = f"ARIMA({int(r['p'])},{int(r['d'])},{int(r['q'])})"
            model_names.append(name)

        out_df = pd.DataFrame({
            "model": model_names,
            "constant": top_df["constant"].astype(bool).values,
            "criterion": top_df["criterion"].values,
            "score": top_df["score"].round(4).values,
            "aic": top_df["aic"].round(4).values,
            "aicc": top_df["aicc"].round(4).values,
            "bic": top_df["bic"].round(4).values,
        })
        return out_df

    def top_modes(self, n: int = 5) -> pd.DataFrame:
        """Alias for top_models(n)."""
        return self.top_models(n=n)

    @property
    def best_orders(self) -> Dict[str, Any]:
        """
        Return a parameter dictionary for the best selected model.
        The returned dictionary can be unpacked directly into peshbeen's arima:
        ``arima(target_col=..., **model.best_orders)``.
        """
        return self.get_orders(rank=1)

    def get_orders(
        self,
        rank: int = 1,
        top_n: Optional[int] = None,
        include_score: bool = False
    ) -> Union[Dict[str, Any], List[Dict[str, Any]]]:
        """
        Return model orders ('order', 'seasonal_order', 'seasonal_length', 'include_constant')
        in a dictionary for the best model (default rank=1) or any runner-up model (rank=2, 3, ...),
        or a list of dictionaries for the top n models.

        The returned dictionary keys match the constructor arguments of ``arima`` so it can be
        unpacked directly: ``arima(target_col=..., **model.get_orders(rank=2))``.

        Parameters
        ----------
        rank : int, default 1
            1-based rank of the candidate model (1 = best model, 2 = second best, etc.).
        top_n : int, optional
            If provided, returns a list of parameter dictionaries for the top n models instead of a single dictionary.
        include_score : bool, default False
            Whether to also include criterion scores ('criterion', 'score', 'aic', 'aicc', 'bic')
            in the dictionary. Set to False (default) to unpack directly into ``arima``.

        Returns
        -------
        dict or list of dict
            Dictionary or list of dictionaries containing:
            - 'order': tuple of int (p, d, q)
            - 'seasonal_order': tuple of int (P, D, Q)
            - 'seasonal_length': int or None
            - 'include_constant': bool
        """
        if self.search_results_ is None or len(self.search_results_) == 0:
            raise ValueError("Model has not been fitted yet. Call .fit(df) before calling get_orders().")

        def _row_to_dict(r: pd.Series) -> Dict[str, Any]:
            s_len = int(r["s"]) if r["s"] > 1 else None
            d = {
                "order": (int(r["p"]), int(r["d"]), int(r["q"])),
                "seasonal_order": (int(r["P"]), int(r["D"]), int(r["Q"])),
                "seasonal_length": s_len,
                "include_constant": bool(r["constant"]),
            }
            if include_score:
                d["criterion"] = str(r["criterion"])
                d["score"] = float(np.round(r["score"], 4))
                d["aic"] = float(np.round(r["aic"], 4))
                d["aicc"] = float(np.round(r["aicc"], 4))
                d["bic"] = float(np.round(r["bic"], 4))
            return d

        if top_n is not None:
            n_models = min(int(top_n), len(self.search_results_))
            return [_row_to_dict(self.search_results_.iloc[i]) for i in range(n_models)]

        # 1-based rank (or 0 if user passed 0)
        idx = max(int(rank) - 1, 0)
        if idx >= len(self.search_results_):
            raise IndexError(f"Rank {rank} out of range; only {len(self.search_results_)} models evaluated.")

        return _row_to_dict(self.search_results_.iloc[idx])

    def cross_validate(
        self,
        df: pd.DataFrame,
        cv_split: int,
        test_size: int,
        metrics: List[Callable],
        step_size: int = 1,
        h_split_point: Optional[int] = None,
        refit_selection: bool = False
    ) -> pd.DataFrame:
        """
        Run time-series cross-validation adhering to the peshbeen model validation contract.

        Parameters
        ----------
        df : pd.DataFrame
            DataFrame containing target_col and features.
        cv_split : int
            Number of time series cross-validation folds.
        test_size : int
            Horizon length for each validation test set.
        metrics : list of callable
            Evaluation metrics (e.g. [MAE, RMSE, WMAPE]).
        step_size : int, default 1
            Step size to roll forward between splits.
        h_split_point : int, optional
            Optional split point to report short vs long horizon metrics.
        refit_selection : bool, default False
            If True, re-runs full auto_arima model selection on each training split.
            If False (default), selects the best order once on initial split and refits that order on subsequent folds.

        Returns
        -------
        pd.DataFrame
            Fold-by-fold predictions and errors. Aggregated scores are stored in `self.cv_summary`.
        """
        cv_df_ = pd.DataFrame()
        tscv = SplitTimeSeries(n_splits=cv_split, test_size=test_size, step_size=step_size)
        metrics_dict = {m.__name__: [] for m in metrics}
        if h_split_point is not None:
            metrics_dict1 = {m.__name__: [] for m in metrics}
            metrics_dict2 = {m.__name__: [] for m in metrics}

        selected_order = None
        selected_seas_order = None
        selected_c = None

        for idx, (train_index, test_index) in enumerate(tscv.split(df)):
            train, test = df.iloc[train_index], df.iloc[test_index]
            x_test = test.drop(columns=[self.target_col])
            y_test = np.array(test[self.target_col])

            if refit_selection or selected_order is None:
                self.fit(train)
                selected_order = self.order_
                selected_seas_order = self.seasonal_order_
                selected_c = self.include_constant_
            else:
                # Refit pre-selected best model structure on expanding train window
                mod = arima(
                    target_col=self.target_col,
                    order=selected_order,
                    seasonal_order=selected_seas_order,
                    seasonal_length=self.s if self.s > 1 else None,
                    lag_transform=self.lag_transform,
                    trend=self.trend,
                    pol_degree=self.pol_degree,
                    ets_params=self.ets_params,
                    change_points=self.change_points,
                    box_cox=self.box_cox,
                    box_cox_biasadj=self.box_cox_biasadj,
                    cat_variables=self.cat_variables,
                    categorical_encoder=self.categorical_encoder,
                    target_encode=self.target_encode,
                    include_constant=selected_c,
                    maxiter=self.maxiter,
                    tol=self.tol
                )
                mod.fit(train)
                self.best_model_ = mod

            exog_t = x_test if x_test.shape[1] > 0 else None
            bb_forecast = self.forecast(test_size, exog_t)

            for m in metrics:
                if m.__name__ in ["MASE", "SMAE", "SRMSE", "RMSSE"]:
                    eval_val = m(y_test, bb_forecast, train[self.target_col])
                else:
                    eval_val = m(y_test, bb_forecast)
                metrics_dict[m.__name__].append(eval_val)

            if h_split_point is not None and isinstance(h_split_point, int):
                y_test_1, y_test_2 = y_test[:h_split_point], y_test[h_split_point:]
                bb_forecast_1, bb_forecast_2 = bb_forecast[:h_split_point], bb_forecast[h_split_point:]
                for m in metrics:
                    if m.__name__ in ["MASE", "SMAE", "SRMSE", "RMSSE"]:
                        eval_val1 = m(y_test_1, bb_forecast_1, np.array(train[self.target_col]))
                        eval_val2 = m(y_test_2, bb_forecast_2, np.array(train[self.target_col]))
                    else:
                        eval_val1 = m(y_test_1, bb_forecast_1)
                        eval_val2 = m(y_test_2, bb_forecast_2)
                    metrics_dict1[m.__name__].append(eval_val1)
                    metrics_dict2[m.__name__].append(eval_val2)

            split_results = {
                "cutoff": np.repeat(train.index[-1], len(test)),
                "fold_index": test.index,
                "horizon": np.arange(1, len(test) + 1),
                "split": np.repeat(f"fold_{idx+1}", len(test)),
                "y_true": y_test,
                "y_pred": bb_forecast,
            }
            cv_df_ = pd.concat([cv_df_, pd.DataFrame(split_results)], ignore_index=True)

        overall_performance = pd.DataFrame(
            [[m.__name__, np.mean(metrics_dict[m.__name__])] for m in metrics],
            columns=["eval_metric", "overall_score"],
        )
        if h_split_point is not None and isinstance(h_split_point, int):
            perf_1_df = pd.DataFrame(
                [[m.__name__, np.mean(metrics_dict1[m.__name__])] for m in metrics],
                columns=["eval_metric", f"1-{h_split_point}"],
            )
            perf_2_df = pd.DataFrame(
                [[m.__name__, np.mean(metrics_dict2[m.__name__])] for m in metrics],
                columns=["eval_metric", f"{h_split_point+1}-{test_size}"],
            )
            overall_performance = (
                overall_performance
                .merge(perf_1_df, on="eval_metric")
                .merge(perf_2_df, on="eval_metric")
            )

        self.cv_summary = overall_performance
        return cv_df_

    def get_name(self) -> str:
        """Return model identifier name."""
        return "auto_arima"

    def set_params(self, params: Optional[Dict[str, Any]] = None, **kwargs) -> "auto_arima":
        """Update auto_arima search parameters."""
        updates = {}
        if params is not None:
            if not isinstance(params, dict):
                raise TypeError("params must be a dictionary")
            updates.update(params)
        updates.update(kwargs)

        for k, v in updates.items():
            if hasattr(self, k):
                setattr(self, k, v)
        return self

    def copy(self) -> "auto_arima":
        """Return deep copy of auto_arima instance."""
        return copy.deepcopy(self)


# %% auto #0
__all__ = ["auto_arima", "check_stationarity_d"]


# Automatic ARIMA Modeling via Hyndman-Khandakar Stepwise Algorithm

This module implements the **Hyndman-Khandakar (2008)** automatic ARIMA model selection algorithm within the unified **peshbeen** pipeline:

1. **Stationarity Determination ($):**
   - If $ is not specified (), repeated **KPSS tests** (default) are applied at significance level $\alpha = 0.05$.
   - The null hypothesis $ is level-stationarity. If rejected ( < \alpha$), the series is differenced ( \leftarrow d + 1$) and tested again up to  (default 2).

2. **Seasonal Differencing ($):**
   - If  > 1$ and $ is not specified (), $ is included in the candidate search space (zsh \le D \le \text{max\_D}$, typically $\{0, 1\}$).

3. **Stepwise Search ():**
   - **Step 2(a) Initial Models:**
     - $\text{ARIMA}(0, d, 0)(0, D_{\text{init}}, 0)[s]$
     - $\text{ARIMA}(2, d, 2)(1, D_{\text{init}}, 1)[s]$
     - $\text{ARIMA}(1, d, 0)(1, D_{\text{init}}, 0)[s]$
     - $\text{ARIMA}(0, d, 1)(0, D_{\text{init}}, 1)[s]$
     - A constant $ is included unless  + D \ge 2$. If  + D \le 1$, $\text{ARIMA}(0, d, 0)$ without constant is also evaluated.
   - **Step 2(b) Current Best:** Model with lowest $\text{AICc}$ becomes current model.
   - **Step 2(c) Neighborhood Variations:**
     - Vary $ and/or $ by $\pm 1$.
     - Vary $ and/or $ by $\pm 1$ (if seasonal).
     - Vary $ by $\pm 1$ (if $ was None).
     - Toggle constant term $ (include/exclude).
   - **Step 2(d) Iteration:** The best variation becomes the new current model. Repeat until no neighbor achieves a lower $\text{AICc}$.

4. **Exhaustive Grid Search ():**
   - Full combinatorial evaluation over zsh \le p \le \text{max\_p}$, zsh \le q \le \text{max\_q}$, zsh \le P \le \text{max\_P}$, zsh \le Q \le \text{max\_Q}$,  \in \text{D\_space}$, and constant terms. Orders are strictly bounded by user-defined maximums.


In [2]:
#| hide
# 1. Stationarity Testing for Integration Order d
import numpy as np
from peshbeen.datasets import load_airline_passengers, load_nile, load_wales_admissions

for name, load_fn in [
    ('Airline Passengers', load_airline_passengers),
    ('Nile River Flow', load_nile),
    ('Wales NHS Admissions', load_wales_admissions)
]:
    df = load_fn()
    col = df.columns[0]
    d_sel, logs = check_stationarity_d(df[col].values, max_d=2, alpha=0.05, test='kpss')
    print(f"{name:<22}: determined d = {d_sel} | logs = {logs}")


Airline Passengers    : determined d = 1 | logs = [{'d': 0, 'stat': 1.6513122354165206, 'pval': 0.01, 'action': 'difference'}, {'d': 1, 'stat': 0.023897614400183967, 'pval': 0.1, 'action': 'stationary_stop'}]
Nile River Flow       : determined d = 1 | logs = [{'d': 0, 'stat': 0.8691205593783892, 'pval': 0.01, 'action': 'difference'}, {'d': 1, 'stat': 0.11710083018670835, 'pval': 0.1, 'action': 'stationary_stop'}]
Wales NHS Admissions  : determined d = 2 | logs = [{'d': 0, 'stat': 3.4790611725614835, 'pval': 0.01, 'action': 'difference'}, {'d': 1, 'stat': 0.49467195612064957, 'pval': 0.04286667654940325, 'action': 'difference'}, {'d': 2, 'stat': 0.0818848289444452, 'pval': 0.1, 'action': 'max_d_reached'}]


In [3]:
#| hide
# 2. Hyndman-Khandakar Stepwise Search on Seasonal Airline Passengers
from peshbeen.datasets import load_airline_passengers
from peshbeen.models.arima import arima

df_air = load_airline_passengers()
col_air = df_air.columns[0]

# Fit auto_arima using Hyndman-Khandakar stepwise search (s=12, d=None, D=None)
model_stepwise = auto_arima(
    target_col=col_air,
    seasonal_length=12,
    max_p=3,
    max_q=3,
    max_P=2,
    max_Q=2,
    stepwise=True,
    trace=False
)
model_stepwise.fit(df_air)

print(model_stepwise.summary())

# Report top 5 candidate models using top_models(n)
print()
print("Top 5 Candidate Models:")
print(model_stepwise.top_models(5))

# Extract parameter dictionaries for best and runner-up models using get_orders()
print()
print("Best Model Parameters (rank 1):")
print(model_stepwise.get_orders(rank=1))
print("Second Best Model Parameters (rank 2):")
print(model_stepwise.get_orders(rank=2))
print("Top 3 Model Dictionaries with Scores (top_n=3):")
for i, ord_dict in enumerate(model_stepwise.get_orders(top_n=3, include_score=True), start=1):
    print(f"  Rank {i}: {ord_dict}")

# Demonstrate direct unpacking into native arima model
second_best_orders = model_stepwise.get_orders(rank=2)
direct_mod = arima(target_col=col_air, **second_best_orders)
direct_mod.fit(df_air)
print()
print(f"Directly fitted runner-up arima model with AICc = {direct_mod.aicc:.4f}")

# Multi-step 12-month ahead recursive forecast from auto_arima
fc_12 = model_stepwise.forecast(H=12)
print()
print("12-Month Out-of-Sample Forecast:")
print(np.round(fc_12, 2))

                  Auto ARIMA Model Selection Summary
Selection Algorithm:    Hyndman-Khandakar Stepwise Search
Stationarity Test:      KPSS (alpha = 0.05) -> d = 1
Selected Best Model:    ARIMA(3,1,3) (2,1,1)[12]
Include Constant:       False
Optimized Criterion:    AICC = 804.4437
Information Criteria:   AIC = 802.6103 | AICc = 804.4437 | BIC = 829.0542
Models Evaluated:       40 models
--------------------------------------------------------------------------------------
Best Model Parameter Estimates:
Unified Peshbeen ARIMA Model Results: SARIMAX(3,1,3)x(2,1,1,12)
No. Observations: 144          Log-Likelihood: -391.305     AIC: 802.610   
Sigma^2:          108.5512     Fit Time (s):   0.0411       BIC: 829.054   
  Parameter           Estimate     Std.Err      t-stat       P>|z|       [0.025   0.975]
--------------------------------------------------------------------------------------
  ar.L1                -0.3867      0.1021      -3.788      0.0002 ***  [ -0.587  -0.187]
  ar.L2 

In [4]:
#| hide
# 3. Exhaustive Grid Search (stepwise=False) with Strictly Bounded Maximum Orders
# For example: max_p=2, max_q=2, max_P=1, max_Q=1. No order can exceed maximums.
model_grid = auto_arima(
    target_col=col_air,
    seasonal_length=12,
    max_p=2,
    max_q=2,
    max_P=1,
    max_Q=1,
    stepwise=False,
    trace=False
)
model_grid.fit(df_air)

print(f"Grid Selected Order: ARIMA{model_grid.order_} x {model_grid.seasonal_order_}[12], constant={model_grid.include_constant_}")
print(f"Grid Minimum AICc:   {model_grid.aicc:.4f}")
print(f"Models Evaluated:   {model_grid.n_models_evaluated_}")

# Verify that orders never exceed maximums
res = model_grid.search_results_
assert res['p'].max() <= 2, 'p exceeded max_p'
assert res['q'].max() <= 2, 'q exceeded max_q'
assert res['P'].max() <= 1, 'P exceeded max_P'
assert res['Q'].max() <= 1, 'Q exceeded max_Q'
print(f"Search Order Bounds Verified: max(p)={res['p'].max()}, max(q)={res['q'].max()}, max(P)={res['P'].max()}, max(Q)={res['Q'].max()}")


Grid Selected Order: ARIMA(2, 1, 1) x (1, 1, 0)[12], constant=False
Grid Minimum AICc:   918.6427
Models Evaluated:   108
Search Order Bounds Verified: max(p)=2, max(q)=2, max(P)=1, max(Q)=1


In [6]:
#| hide
# 4. User-Specified d and D (Bypassing Stationarity Check and Restricting Seasonal D)
model_fixed = auto_arima(
    target_col=col_air,
    seasonal_length=12,
    d=1,
    D=1,
    max_p=2,
    max_q=2,
    max_P=1,
    max_Q=1,
    stepwise=True
)
model_fixed.fit(df_air)
print(f"Fixed (d=1, D=1) Selected: ARIMA{model_fixed.order_} x {model_fixed.seasonal_order_}[12], AICc={model_fixed.aicc:.4f}")


Fixed (d=1, D=1) Selected: ARIMA(2, 1, 1) x (1, 1, 0)[12], AICc=918.6427


In [5]:
#| hide
# 5. Exogenous Variables, Categorical Features, and Lag Transforms under Peshbeen Pipeline
from peshbeen.datasets import load_wales_admissions
from sklearn.preprocessing import OneHotEncoder
from peshbeen.transformations import rolling_mean

wales_df = load_wales_admissions().iloc[-120:].copy()
wales_df['day_of_week'] = wales_df.index.dayofweek.astype(str)

model_exog = auto_arima(
    target_col='admissions',
    seasonal_length=7,
    max_p=2,
    max_q=2,
    max_P=1,
    max_Q=1,
    cat_variables=['day_of_week'],
    categorical_encoder=OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),
    lag_transform=[rolling_mean(window_size=3, shift=1)],
    stepwise=True
)
model_exog.fit(wales_df)

print(f"Exogenous Model Selected: ARIMA{model_exog.order_} x {model_exog.seasonal_order_}[7], AICc={model_exog.aicc:.4f}")
exog_future = wales_df.drop(columns=['admissions']).iloc[-7:]
fc_exog = model_exog.forecast(7, exog=exog_future)
print("7-Day Forecast with Exogenous Regressors:", np.round(fc_exog, 1))


Exogenous Model Selected: ARIMA(0, 0, 2) x (0, 1, 1)[7], AICc=1269.5026
7-Day Forecast with Exogenous Regressors: [8829.6 8861.4 8864.8 8914.4 8905.3 8886.2 8813.3]


In [6]:
#| hide
# 6. Time-Series Cross Validation adhering to Peshbeen Contract
from peshbeen.metrics import MAE, RMSE

wales_sub = load_wales_admissions().iloc[-150:]
model_cv = auto_arima(
    target_col='admissions',
    seasonal_length=7,
    max_p=2,
    max_q=2,
    max_P=1,
    max_Q=1,
    stepwise=True
)

cv_df = model_cv.cross_validate(
    df=wales_sub,
    cv_split=3,
    test_size=14,
    metrics=[MAE, RMSE],
    step_size=14
)

print("Cross-Validation Performance Summary:")
print(model_cv.cv_summary)


Cross-Validation Performance Summary:
  eval_metric  overall_score
0         MAE      61.311990
1        RMSE      81.352752
